# 03 · Ranking: tf-idf lnc.ltc, BM25, BM25F, authority g(d), adaptive λ

In [ ]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
from scholar_ir.config import load_config, p
cfg = load_config()  # config.yaml at the repo root (or $SCHOLAR_IR_CONFIG)
R = p(cfg, "results_dir")

In [ ]:
from scholar_ir.engine import SearchEngine
engine = SearchEngine.load(cfg)
index, papers = engine.index, engine.papers
print(f"{index.N:,} papers, years {papers.year.min()}-{papers.year.max()}, tuned params loaded: {bool(engine.tuned)}")

In [ ]:
from scholar_ir.explain import format_explain
q = 'dense passage retrieval for question answering'
print(format_explain(engine.search(q, system='S1', k=3, explain=True)))

In [ ]:
print(format_explain(engine.search(q, system='S3', k=3, explain=True)))

## Heap-based top-K equals a full sort

In [ ]:
from scholar_ir.ranking import heap_topk, query_terms, bm25f
from scholar_ir.query_parser import parse
import time
pq = parse(q)
rc = engine.cfg['ranking']['bm25f']
acc, touched = bm25f(index, query_terms(index, pq), rc['k1'], rc['w'], rc['b'])
docs = np.flatnonzero(touched); t0 = time.perf_counter(); h = heap_topk(docs.tolist(), acc[docs].tolist(), 10); t1 = time.perf_counter()
s = sorted(zip(docs.tolist(), acc[docs].tolist()), key=lambda x: (-x[1], x[0]))[:10]; t2 = time.perf_counter()
print(f'J={len(docs)} scored docs; heap == sort: {h == s}; heap {1e3*(t1-t0):.1f} ms vs sort {1e3*(t2-t1):.1f} ms')

## Static quality g(d): raw citations are age-biased, cohort percentiles are not

In [ ]:
a = engine.auth.merge(papers[['doc_id']], on='doc_id')
by = a[a.has_citation_data].groupby('year')[['cited_by_count','g_raw','g_per_year','g_cohort','g_pagerank_cohort']].median()
by.round(3)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,3.2))
for c, lab in [('g_raw','A0 raw'), ('g_per_year','A1 per-year'), ('g_cohort','A2 cohort percentile')]:
    ax.plot(by.index, by[c], marker='o', ms=3, label=lab)
ax.set_ylabel('median g(d)'); ax.set_xlabel('first-submission year'); ax.legend(); ax.set_title('Authority by publication year')
plt.show()

## Query-adaptive λ: features and λ for a few queries

In [ ]:
rows = []
for t in ['seminal work on word embeddings', 'survey of recommender systems', 'LLM agents that use tools',
          'latest work on privacy attacks in federated learning', 'knowledge graph embedding']:
    r = engine.search(t, system='S7', k=10); f = r.features
    rows.append({'query': t, 'found_cue': f.found_cue, 'recent_cue': f.recent_cue, 'idf_z': round(f.idf_z,2),
                 'len_z': round(f.len_z,2), 'flatness': round(f.flatness,2), 'p_found': round(r.p_found,3), 'lambda': round(r.lam,3)})
pd.DataFrame(rows)